# Query 7: CTEs, Temporary Tables, and Data Cleaning

Run `ddl.ipynb` and `dml.ipynb` first. This notebook names intermediate query results, checks data quality, and creates a temporary summary table.

In [ ]:
%load_ext sql
%sql sqlite:///food-analysis/food_analysis.db

## Use a CTE for monthly partner totals

A common table expression (CTE) is a named query that exists for one statement. It can make a multi-step query easier to read.

In [ ]:
%%sql
WITH monthly_partner_totals AS (
    SELECT strftime('%Y-%m', order_placed_at) AS order_month,
           partner,
           COUNT(*) AS delivered_orders,
           ROUND(SUM(order_amount), 2) AS delivered_amount
    FROM orders
    WHERE status = 'delivered'
    GROUP BY strftime('%Y-%m', order_placed_at), partner
)
SELECT order_month, partner, delivered_orders, delivered_amount
FROM monthly_partner_totals
WHERE delivered_amount >= 5000
ORDER BY order_month DESC, delivered_amount DESC;

## Check for missing or blank order fields

These checks report possible quality issues without changing the source data. `NULLIF(TRIM(...), '')` treats blank text as missing.

In [ ]:
%%sql
SELECT COUNT(*) AS total_orders,
       SUM(CASE WHEN order_placed_at IS NULL OR TRIM(order_placed_at) = '' THEN 1 ELSE 0 END) AS missing_order_dates,
       SUM(CASE WHEN NULLIF(TRIM(status), '') IS NULL THEN 1 ELSE 0 END) AS missing_statuses,
       SUM(CASE WHEN order_amount IS NULL OR order_amount < 0 THEN 1 ELSE 0 END) AS invalid_amounts
FROM orders;

## Preview normalized customer areas

Use `TRIM` to remove extra spaces and `COALESCE` to give missing values a consistent label. This is a query-only preview; it does not update the table.

In [ ]:
%%sql
WITH cleaned_customers AS (
    SELECT customer_id,
           customer_name,
           COALESCE(NULLIF(TRIM(area), ''), 'Unknown area') AS clean_area
    FROM customers
)
SELECT clean_area, COUNT(*) AS customer_count
FROM cleaned_customers
GROUP BY clean_area
ORDER BY customer_count DESC, clean_area;

## Save a top-product summary in a temporary table

A temporary table is available only in the current database connection. Drop it before creating it so the notebook can be rerun.

In [ ]:
%%sql
DROP TABLE IF EXISTS temp.top_product_sales;

In [ ]:
%%sql
CREATE TEMP TABLE top_product_sales AS
SELECT p.product_id, p.product_name,
       SUM(oi.quantity) AS units_sold,
       ROUND(SUM(oi.line_amount), 2) AS item_sales
FROM order_items AS oi
JOIN products AS p ON p.product_id = oi.product_id
JOIN orders AS o ON o.order_id = oi.order_id
WHERE o.status = 'delivered'
GROUP BY p.product_id, p.product_name;

In [ ]:
%%sql
SELECT product_name, units_sold, item_sales
FROM top_product_sales
ORDER BY item_sales DESC
LIMIT 5;

In [ ]:
%%sql
DROP TABLE temp.top_product_sales;